# 🐭 go or nogo? — the real neuron, step by step

**Time:** ~45 minutes &nbsp;•&nbsp; **Goal:** run the analysis from notebook 1 on the **real** recording, and answer the experiment's question.

## The experiment

This data comes from **Miguel Maravall's lab here at Sussex** ([Bale et al., 2021, *Current Biology*](https://doi.org/10.1016/j.cub.2020.10.059)). A mouse feels one of two whisker "words" — **go** (lick, and get water) or **nogo** (don't lick) — while a microscope records one neuron's activity.

**The question:** does this neuron tell go from nogo?

## How this notebook works

These are the **same eight steps** as notebook 1, in the same order — 📎 each one says which step to look back at. Only the sizes change:

| | notebook 1 | here |
|---|---|---|
| frames | 50 | 21,238 |
| each word lasts | 3 frames | **9** frames |
| window | 2 before, 5 after | **10** before, **30** after |

Each 🔬 **Write it** gives you a code skeleton: replace every `___` and run the cell. The self-check underneath tells you ✅ or ❌. If you see `name '___' is not defined`, you missed a blank.

## Our data

Run this first. It loads **neuron 1** — the one from the lecture — and downloads **neuron 2** for the finale.

Each column of the file is loaded with `np.loadtxt(..., usecols=n)`: column 1 is `dff`, column 2 is `go`. You load **nogo** yourself, just below.

Printing a big array shows only the start and the end, with `...` in the middle.

In [ ]:
# Run this first. It loads two neurons' recordings from the lecture —
# from data/ if you downloaded them, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
dff = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=1)
go  = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=2)

FRAME_RATE = 10.8   # frames per second
print("dff:", dff)
print("go: ", go)

## Step 1 — Meet the arrays &nbsp; 📎 *notebook 1, step 1*

### 🔬 Write it

**Your task:**

1. Load the **nogo** column — column **3** — into `nogo`. Copy the `go` line from the data cell and change two things.
2. Store how many frames there are in `n_frames`.
3. Store how long the recording is in **minutes**, rounded to 1 decimal place, in `minutes`. Frames ÷ `FRAME_RATE` gives seconds.

In [ ]:
# TODO — replace every ___ and run
nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=___)
n_frames = dff.___
minutes = round(n_frames / ___ / 60, 1)
print(n_frames, "frames =", minutes, "minutes")


# --- self-check (do not edit) ---
assert "nogo" in globals() and np.shape(nogo) == (21238,), "❌ Load column 3 into nogo"
assert set(np.unique(nogo)) == {0.0, 1.0} and not np.array_equal(nogo, go), "❌ nogo should be column 3 — a different 0/1 column from go"
assert n_frames == 21238, f"❌ n_frames should be 21238, got {n_frames} — dff.size"
assert minutes == 32.8, f"❌ minutes should be 32.8, got {minutes} — n_frames / FRAME_RATE / 60, rounded to 1"
print("✅ Correct — over half an hour of one neuron, and nogo is loaded")

## Step 2 — A first try: a mask &nbsp; 📎 *notebook 1, step 2*

In notebook 1 the mask found nothing, because the neuron answers **after** the word. Does the real neuron do the same?

In [ ]:
# DEMO - activity while go is on, against the whole recording
print("mean while go is on:", round(dff[go == 1].mean(), 3))
print("mean overall:       ", round(dff.mean(), 3))

### 🔬 Write it

**Your task:** store the mean of `dff` **while nogo is on**, rounded to 3 decimal places, in `nogo_on_mean`.

In [ ]:
# TODO — replace the ___
nogo_on_mean = round(dff[___].mean(), 3)
print("mean while nogo is on:", nogo_on_mean)


# --- self-check (do not edit) ---
assert "nogo_on_mean" in globals(), "❌ No variable called nogo_on_mean yet"
assert nogo_on_mean != round(dff.mean(), 3), "❌ That's the mean of every frame — put the mask in the brackets: dff[nogo == 1]"
assert nogo_on_mean != round(dff[go == 1].mean(), 3), "❌ That's the go answer — make the mask from nogo"
assert nogo_on_mean == 0.007, f"❌ nogo_on_mean should be 0.007, got {nogo_on_mean}"
print("✅ Correct — both words give about 0, below the overall mean of 0.058. Same as notebook 1: the mask misses the response.")

## Step 3 — Where is the word on? &nbsp; 📎 *notebook 1, step 3*

`np.where(...)[0]` gives the frame numbers where the mask is `True`. Here the blocks are **9** frames long.

In [ ]:
# DEMO - the frame numbers where go is on
go_on = np.where(go == 1)[0]
print(go_on[:20])
print(len(go_on), "frames with go on")

### 🔬 Write it

**Your task:** make `nogo_on` — every frame number where **nogo** is on.

In [ ]:
# TODO — replace the ___  (remember the [0]!)
nogo_on = np.where(___)[0]
print(nogo_on[:20])


# --- self-check (do not edit) ---
assert "nogo_on" in globals(), "❌ No variable called nogo_on yet"
assert not isinstance(nogo_on, tuple), "❌ nogo_on is a tuple — add [0] after np.where(...)"
assert len(nogo_on) == 2376, f"❌ nogo should be on in 2376 frames, got {len(nogo_on)} — np.where(nogo == 1)[0]"
print("✅ Correct — 2376 frames: 264 nogo words of 9 frames each")

## Step 4 — Where does each trial start? &nbsp; 📎 *notebook 1, step 4*

The first frame of each block of 9 — so every **9th** position.

In [ ]:
# DEMO - the first frame of each go trial
go_starts = go_on[::9]
print(go_starts[:5])
print(len(go_starts), "go trials")

### 🔬 Write it

**Your task:** make `nogo_starts` from `nogo_on`. Then look at the **first** one — as in notebook 1, it is very close to the beginning.

In [ ]:
# TODO — replace the ___
nogo_starts = nogo_on[___]
print(len(nogo_starts), "nogo trials; the first starts at frame", nogo_starts[0])


# --- self-check (do not edit) ---
assert "nogo_starts" in globals(), "❌ No variable called nogo_starts yet"
assert len(nogo_starts) != 2376, "❌ That's every nogo frame — keep every 9th with [::9]"
assert len(nogo_starts) != 792, "❌ Every 3rd? In the real data each word lasts 9 frames: [::9]"
assert len(nogo_starts) == 264, f"❌ There should be 264 nogo trials, got {len(nogo_starts)}"
assert nogo_starts[0] == 2, f"❌ The first nogo trial should start at frame 2, got {nogo_starts[0]}"
print("✅ Correct — 264 nogo trials, and the first starts at frame 2. Keep that in mind for Step 6.")

## Step 5 — Cut one window round a start &nbsp; 📎 *notebook 1, step 5*

10 frames **before** the start, 30 from it on: `dff[start - 10 : start + 30]` — 40 frames, about 3.7 seconds. In the window, the word starts at **position 10**.

In [ ]:
# DEMO - the window round the first go trial
start = go_starts[0]
window = dff[start - 10 : start + 30]
print("starts at frame", start, "- window of", window.size, "frames")
print(window.round(2))

### 🔬 Write it

**Your task:** cut the window round the **second** go trial into `second_window`.

In [ ]:
# TODO — replace every ___
start = go_starts[___]
second_window = dff[start - ___ : start + ___]
print(second_window.size, "frames")


# --- self-check (do not edit) ---
assert "second_window" in globals(), "❌ No variable called second_window yet"
assert np.size(second_window) == 40, f"❌ A window should be 40 frames — 10 before and 30 after — got {np.size(second_window)}"
assert np.allclose(second_window, dff[go_starts[1] - 10 : go_starts[1] + 30]), "❌ That's not the window round the second go start — go_starts[1]"
print("✅ Correct — frames", go_starts[1] - 10, "to", go_starts[1] + 29)

## Step 6 — Cut them all, and stack them &nbsp; 📎 *notebook 1, step 6*

The loop from notebook 1, with the real sizes. The result is a grid: **254 rows** (trials) × **40 columns** (moments).

In [ ]:
# DEMO - every go window, stacked into a grid
go_trials = []
for start in go_starts:
    go_trials.append(dff[start - 10 : start + 30])
go_trials = np.array(go_trials)

print(go_trials.shape)
print("one trial:              ", go_trials[0].shape)
print("one moment, every trial:", go_trials[:, 10].shape)

### 🐞 Broken — the trial at frame 2

The same loop for nogo crashes with an **inhomogeneous shape** — the rows aren't all the same length. You met this in notebook 1.

> 💡 The first nogo trial starts at frame 2, so it asks for `dff[-8:32]` — empty. Keep only trials with at least **10** frames before them.

In [ ]:
# 🐞 BROKEN — every nogo window, stacked
nogo_trials = []
for start in nogo_starts:
    nogo_trials.append(dff[start - 10 : start + 30])
nogo_trials = np.array(nogo_trials)
print(nogo_trials.shape)


# --- self-check (do not edit) ---
assert nogo_trials.shape == (263, 40), f"❌ nogo_trials should be (263, 40), got {nogo_trials.shape} — skip trials that start before frame 10"
print("✅ Fixed — 263 nogo trials, each 40 frames long")

## Step 7 — Average down the columns &nbsp; 📎 *notebook 1, step 7*

`.mean(axis=0)` collapses the 254 rows into **one average trial**: 40 numbers, one per moment. The noise cancels; the response stays.

### 🔬 Write it

**Your task:**

1. Make `go_response` — the average go trial.
2. Store its peak **from the start onwards** — position 10 on — rounded to 3 decimal places, in `go_peak`.

In [ ]:
# TODO — replace every ___
go_response = go_trials.mean(axis=___)
go_peak = round(go_response[___:].max(), 3)
print(go_response.shape, "-> peak", go_peak)


# --- self-check (do not edit) ---
assert "go_response" in globals() and "go_peak" in globals(), "❌ Make go_response and go_peak"
assert np.shape(go_response) != (254,), "❌ go_response has one value per TRIAL — that's axis=1. You want one per moment: axis=0"
assert np.shape(go_response) == (40,), f"❌ go_response should have 40 values, got shape {np.shape(go_response)}"
assert go_peak == 0.29, f"❌ go_peak should be 0.29, got {go_peak}"
print("✅ Correct — the average go trial peaks at", go_peak)

## Step 8 — go against nogo: the answer &nbsp; 📎 *notebook 1, step 8*

### 🔬 Write it

**Your task:**

1. Make `nogo_response` and `nogo_peak` — the same as Step 7, for nogo.
2. Store how many times bigger the go peak is, rounded to 1 decimal place, in `ratio`.

In [ ]:
# TODO — replace every ___
nogo_response = ___
nogo_peak = ___
ratio = round(go_peak / nogo_peak, 1)
print("go", go_peak, "| nogo", nogo_peak, "| go is", ratio, "times bigger")


# --- self-check (do not edit) ---
assert "nogo_response" in globals() and "nogo_peak" in globals() and "ratio" in globals(), "❌ Make nogo_response, nogo_peak and ratio"
assert np.shape(nogo_response) == (40,), f"❌ nogo_response should have 40 values — nogo_trials.mean(axis=0). Got shape {np.shape(nogo_response)}"
assert nogo_peak == 0.057, f"❌ nogo_peak should be 0.057, got {nogo_peak} — nogo_response[10:].max(), rounded to 3"
assert ratio == 5.1, f"❌ ratio should be 5.1, got {ratio}"
print("✅ Correct — the real neuron responds five times more after go than after nogo")

## Draw it

Run this — plotting comes properly in Lab 7. The x-axis is **seconds from the start of the word**: `np.arange(-10, 30)` numbers the 40 frames from −10 to 29, and dividing by the frame rate turns them into seconds.

In [ ]:
# Run me - one trial, and the two averages
seconds = np.arange(-10, 30) / FRAME_RATE

plt.figure(figsize=(8, 3.5))
plt.plot(seconds, go_trials[0], color="lightgrey", label="one go trial")
plt.plot(seconds, go_response, lw=2.5, label=f"go — {len(go_trials)} trials averaged")
plt.plot(seconds, nogo_response, lw=2.5, label=f"nogo — {len(nogo_trials)} trials averaged")
plt.axvline(0, color="black", lw=1)                 # the word starts
plt.axvspan(0, 9 / FRAME_RATE, alpha=0.1)           # ...and plays for 9 frames
plt.xlabel("seconds from the start of the word")
plt.ylabel("activity (dF/F)")
plt.legend()
plt.show()

**Read the picture.** The grey single trial is mostly noise. The averages are not: after **go**, activity climbs as the word ends and stays up; after **nogo** it stays flat. This neuron tells the two words apart — and the mask in Step 2 missed it because it only looked while the word was playing.

> 🧠 **A scientist's doubt.** On go trials the mouse also **licks** and gets **water**. Is the neuron answering the *word*, or the *licking*? This analysis can't tell yet — see *Explore further*.

## 🏆 Finale — one function, a second neuron

Real analysis is running the same steps on the next cell. Wrap Steps 3–7 in a **function** (Lab 4), so one line analyses any neuron.

**Your task:**

1. Complete `response_to(trace, marker)` — it takes a neuron's activity and a go or nogo column, and returns the average trial.
2. Load neuron 2 into `dff_2`, `go_2` and `nogo_2` (columns 1, 2, 3 of `NEURON_2`).
3. Store each neuron's go-peak ÷ nogo-peak, rounded to 1 decimal, in `ratio_1` and `ratio_2`.

> 💡 Inside the function, use `trace` and `marker` — not `dff` and `go` — or it will only ever analyse neuron 1.

In [ ]:
# TODO — replace every ___
def response_to(trace, marker):
    on = np.where(marker == 1)[0]          # Step 3
    starts = on[___]                       # Step 4
    trials = []
    for start in starts:                   # Steps 5 and 6
        if start >= ___:
            trials.append(trace[___])
    trials = np.array(trials)
    return trials.mean(axis=___)           # Step 7

dff_2  = np.loadtxt(NEURON_2, delimiter=",", skiprows=1, usecols=___)
go_2   = ___
nogo_2 = ___

ratio_1 = round(response_to(dff, go)[10:].max() / response_to(dff, nogo)[10:].max(), 1)
ratio_2 = round(response_to(___)[10:].max() / response_to(___)[10:].max(), 1)
print("neuron 1: go is", ratio_1, "times nogo")
print("neuron 2: go is", ratio_2, "times nogo")


# --- self-check (do not edit) ---
assert "response_to" in globals(), "❌ No function called response_to yet"
assert response_to(dff, go) is not None, "❌ response_to gives back None — did you forget to return?"
assert np.shape(response_to(dff, go)) == (40,), f"❌ response_to should return 40 values, got shape {np.shape(response_to(dff, go))}"
assert np.allclose(response_to(dff, go), go_response), "❌ response_to(dff, go) doesn't match go_response from Step 7"
assert not np.allclose(response_to(dff, nogo), go_response), "❌ response_to(dff, nogo) gives the GO answer — inside the function use trace and marker, not dff and go"
assert np.allclose(response_to(dff, nogo), nogo_response), "❌ response_to(dff, nogo) doesn't match nogo_response — did you keep the if start >= 10?"
assert np.shape(dff_2) == (9635,) and np.shape(nogo_2) == (9635,), "❌ Load neuron 2's columns 1, 2 and 3 into dff_2, go_2 and nogo_2"
assert ratio_1 == 5.1, f"❌ ratio_1 should be 5.1, got {ratio_1}"
assert ratio_2 == 1.2, f"❌ ratio_2 should be 1.2, got {ratio_2} — response_to(dff_2, go_2) and response_to(dff_2, nogo_2)"
print("✅ Correct — same code, two neurons, two different answers")

## The verdict

| | go peak | nogo peak | go ÷ nogo |
|---|---|---|---|
| neuron 1 | 0.29 | 0.06 | **5.1** |
| neuron 2 | 1.28 | 1.04 | **1.2** |

**Neuron 1** cares which word it heard. **Neuron 2** fires hard after *both* — it responds to the whiskers being touched, whatever the word. Two cells, two personalities, and the same few lines of numpy found both.

## 🔭 Explore further

- **Words or licks?** Load the `lick` column (`usecols=5`) and look at `dff[lick == 1].mean()` against `dff.mean()`.
- Plot neuron 2: copy the plot cell and use `response_to(dff_2, go_2)` and `response_to(dff_2, nogo_2)`.
- `go_trials.mean(axis=1)` gives one number per trial. Does the neuron respond less towards the end of the session?
- Ready for more? The **advanced** version of this notebook ends with stretch tasks: finding starts with `np.diff`, cutting windows without a loop, and a test for whether a difference is real.

## ✅ You've got it if you can…

- [ ] Load one column of a CSV with `np.loadtxt(..., usecols=...)`
- [ ] Explain why the mean **while** the word plays missed the response
- [ ] Find the start of every trial with `np.where(...)[0][::9]`
- [ ] Cut a window round each start and stack them into a (trials × moments) grid
- [ ] Explain why the trial at frame 2 broke the stacking
- [ ] Use `.mean(axis=0)` to get the average trial
- [ ] Wrap the analysis in a function and rerun it on a second neuron